# AI4Shipwrecks: YOLO Segmentation Training

This notebook prepares the AI4Shipwrecks PNG image and mask dataset for Ultralytics YOLO segmentation, validates the pairs, creates a leakage-resistant train/validation split, trains a model, and evaluates it on the untouched test set.

The original `labels` folders contain binary PNG masks, not YOLO `.txt` labels, so the conversion cell is required.

In [ ]:
# Install the packages used by the notebook.
%pip -q install ultralytics opencv-python-headless scikit-learn matplotlib pandas pyyaml

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import os

# Change this to the folder containing AI4Shipwrecks in your Drive.
# DRIVE_DATASET = Path('/content/drive/MyDrive/AI4Shipwrecks')
# # Example if the folder is inside another Drive directory:
# # DRIVE_DATASET = Path('/content/drive/MyDrive/your-folder/AI4Shipwrecks')

# assert DRIVE_DATASET.exists(), f'Dataset folder not found: {DRIVE_DATASET}'
# print(DRIVE_DATASET)

In [2]:
# Inspect the original dataset before conversion.
from PIL import Image
import pandas as pd

# SPLITS = ['train', 'test']
# for split in SPLITS:
#     image_dir = DRIVE_DATASET / split / 'images'
#     mask_dir = DRIVE_DATASET / split / 'labels'
#     images = sorted(image_dir.glob('*.png'))
#     masks = sorted(mask_dir.glob('*.png'))
#     image_names = {p.stem for p in images}
#     mask_names = {p.stem for p in masks}
#     print(f'{split}: {len(images)} images, {len(masks)} masks')
#     print('  missing masks:', sorted(image_names - mask_names)[:5])
#     print('  orphan masks:', sorted(mask_names - image_names)[:5])
#     assert image_names == mask_names, f'Pair mismatch in {split}'
#     sizes = [Image.open(p).size for p in images[:10]]
#     print('  sample dimensions:', sizes[:3])

# extra_images = list((DRIVE_DATASET / 'extras' / 'terrain' / 'images').glob('*.png'))
# print(f'extras/terrain: {len(extra_images)} images')

In [3]:
# Visual and mask-value sanity check.
import matplotlib.pyplot as plt
import numpy as np

# sample_image = sorted((DRIVE_DATASET / 'train' / 'images').glob('*.png'))[0]
# sample_mask = DRIVE_DATASET / 'train' / 'labels' / sample_image.name
# image = np.array(Image.open(sample_image).convert('L'))
# mask = np.array(Image.open(sample_mask).convert('L'))

# print('image:', image.shape, image.dtype, 'range:', int(image.min()), int(image.max()))
# print('mask:', mask.shape, mask.dtype, 'unique sample:', np.unique(mask)[:20])
# print('mask foreground percentage:', round(float((mask > 0).mean() * 100), 4))

# fig, axes = plt.subplots(1, 2, figsize=(16, 7))
# axes[0].imshow(image, cmap='gray'); axes[0].set_title('Sonar image'); axes[0].axis('off')
# axes[1].imshow(mask, cmap='gray'); axes[1].set_title('Binary mask'); axes[1].axis('off')
# plt.show()

## Configuration

The current dataset has very tall images. Start with `USE_TILES = False` to verify the pipeline. If shipwreck regions are small after resizing, rerun with tiled preparation; the tile extension is provided in the next preparation cell.

In [4]:
import random
import shutil
import cv2
from sklearn.model_selection import GroupShuffleSplit

# WORK_DIR = Path('/content/AI4Shipwrecks_yolo')
# if WORK_DIR.exists():
#     shutil.rmtree(WORK_DIR)
# for path in ['images/train', 'images/val', 'images/test', 'labels/train', 'labels/val', 'labels/test']:
#     (WORK_DIR / path).mkdir(parents=True, exist_ok=True)

# CLASS_NAME = 'shipwreck'
# CLASS_ID = 0
# VAL_SIZE = 0.20
# SEED = 42
# MIN_COMPONENT_AREA = 10
# APPROXIMATION_EPSILON = 0.002
# USE_TILES = False

# # Group by the source name before the final numeric frame suffix, e.g. DM_Wilson_01 -> DM_Wilson.
# def source_group(path):
#     return path.stem.rsplit('_', 1)[0]

# train_images = sorted((DRIVE_DATASET / 'train' / 'images').glob('*.png'))
# groups = [source_group(p) for p in train_images]
# splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
# train_idx, val_idx = next(splitter.split(train_images, groups=groups))
# train_files = [train_images[i] for i in train_idx]
# val_files = [train_images[i] for i in val_idx]
# print('train files:', len(train_files), 'groups:', sorted({source_group(p) for p in train_files}))
# print('val files:', len(val_files), 'groups:', sorted({source_group(p) for p in val_files}))

In [ ]:
def mask_to_yolo_segments(mask_path, label_path):
    # Convert every connected foreground component to one normalized polygon.
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(mask_path)
    binary = np.where(mask > 0, 255, 0).astype(np.uint8)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    height, width = binary.shape
    polygons = []
    for contour in contours:
        if cv2.contourArea(contour) < MIN_COMPONENT_AREA:
            continue
        epsilon = APPROXIMATION_EPSILON * cv2.arcLength(contour, True)
        polygon = cv2.approxPolyDP(contour, epsilon, True).reshape(-1, 2)
        if len(polygon) < 3:
            continue
        normalized = []
        for x, y in polygon:
            normalized.extend([float(x) / width, float(y) / height])
        polygons.append(f'{CLASS_ID} ' + ' '.join(f'{value:.6f}' for value in normalized))
    label_path.write_text('\n'.join(polygons) + ('\n' if polygons else ''), encoding='utf-8')
    return len(polygons)

def copy_and_convert(files, split):
    total_polygons = 0
    empty_masks = 0
    for image_path in files:
        mask_path = image_path.parents[1] / 'labels' / image_path.name
        output_image = WORK_DIR / 'images' / split / image_path.name
        output_label = WORK_DIR / 'labels' / split / f'{image_path.stem}.txt'
        shutil.copy2(image_path, output_image)
        polygons = mask_to_yolo_segments(mask_path, output_label)
        total_polygons += polygons
        empty_masks += int(polygons == 0)
    return len(files), total_polygons, empty_masks

print('train:', copy_and_convert(train_files, 'train'))
print('val:', copy_and_convert(val_files, 'val'))
test_files = sorted((DRIVE_DATASET / 'test' / 'images').glob('*.png'))
print('test:', copy_and_convert(test_files, 'test'))

In [ ]:
# Inspect converted annotations and verify normalized YOLO coordinates.
label_files = sorted((WORK_DIR / 'labels' / 'train').glob('*.txt'))
assert label_files, 'No labels were generated'
for label_file in label_files[:5]:
    for line in label_file.read_text(encoding='utf-8').splitlines():
        values = [float(v) for v in line.split()]
        assert values[0] == CLASS_ID
        assert len(values) >= 7 and len(values[1:]) % 2 == 0
        assert all(0.0 <= v <= 1.0 for v in values[1:])
    print(label_file.name, label_file.read_text(encoding='utf-8')[:160])
print('Annotation validation passed.')

In [5]:
# Create the Ultralytics dataset configuration.
import yaml

# data_yaml = {
#     'path': str(WORK_DIR),
#     'train': 'images/train',
#     'val': 'images/val',
#     'test': 'images/test',
#     'names': {CLASS_ID: CLASS_NAME},
# }
# DATA_YAML = WORK_DIR / 'data.yaml'
# DATA_YAML.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding='utf-8')
# print(DATA_YAML.read_text())

In [6]:
# Train a segmentation model.
from ultralytics import YOLO

# model = YOLO('yolo11n-seg.pt')
# results = model.train(
#     data=str(DATA_YAML),
#     epochs=100,
#     imgsz=1024,
#     batch=2,
#     workers=2,
#     patience=20,
#     device=0,
#     project='/content/drive/MyDrive/AI4Shipwrecks_runs',
#     name='yolo11n_seg_baseline',
#     pretrained=True,
#     seed=SEED,
#     plots=True,
#     exist_ok=True,
# )

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\MANASH RANJAN\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
# Validate on the validation split and evaluate on the untouched test split.
best_model_path = Path('D:\Documents\Manash\COLLAGE\PROJECTS\HACKATHONS\11. ISIH 2026 1\ai\models\AI4Shipwrecks\yolo11n_seg_baseline\weights\best.pt')
best_model = YOLO(str(best_model_path))

val_metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=1024, batch=2, device=0, plots=True)
test_metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=1024, batch=2, device=0, plots=True)
print('Validation metrics:', val_metrics.results_dict)
print('Test metrics:', test_metrics.results_dict)

In [ ]:
# Run qualitative predictions on test images.
prediction_dir = Path('/content/drive/MyDrive/AI4Shipwrecks_runs/predictions')
prediction_dir.mkdir(parents=True, exist_ok=True)
test_sample = sorted((WORK_DIR / 'images' / 'test').glob('*.png'))[:8]
best_model.predict(
    source=[str(p) for p in test_sample],
    imgsz=1024,
    conf=0.25,
    device=0,
    save=True,
    project=str(prediction_dir),
    name='test_samples',
    exist_ok=True,
)
print('Predictions saved under:', prediction_dir)

## Notes for improving the baseline

- If masks are very sparse or objects are small, use overlapping 1024x1024 tiles before conversion and training.
- Keep all tiles from one original source image in the same split.
- Try `yolo11s-seg.pt` after the baseline if GPU memory allows.
- Compare validation and test metrics; a large gap may indicate source-distribution differences or overfitting.
- Inspect prediction images, not only mAP, because sonar masks can be visually difficult even when the metric looks acceptable.
- The `extras/terrain` data is not included in this baseline. Add it only after confirming it uses the same class definition and acquisition distribution.